# UIDAI Aadhaar Data — Curation & Visualization Assignment
#Problem Statement: Unlocking Societal Trends in Aadhaar Enrolment and Updates
This problem is taken from the government hackathon 2026

https://event.data.gov.in/challenge/uidai-data-hackathon-2026/

Identify meaningful patterns, trends, anomalies, or predictive indicators and translate them into clear insights or solution frameworks that can support informed decision-making and system improvements.

Datasets are available here:
https://www.kaggle.com/datasets/ismatkhan121/uidai-data-hackathon-2026-dataset

---
# Part 1 — Dataset Summary

## What it is?

First Identify What is the data is about and fill it here.
Are ethics followed in this dataset?
Anonymized data?

ANS :  the data is about the biometric , demographic and enrollment details of people from different states , classified by their age groups (5-17 , 17 or higher etc..) and dates carried out

Yes , ethics are being followed since none of the people's names , important personal details like identity numbers , addresses etc.. have been revealed in the dataset

Yes the data is fully anonymized since no details are mentioned to narrow down to the people from whom it is collected from .

## What makes this dataset worth studying

Explain why this dataset is given as hackathon problem and why it is important for Data Curation

## Provenance — read before you start

Be precise about what is established and what is not:

Several questions below ask you to verify the participant-reported figures rather than accept
them. **"Their number is wrong, and here is why" is a fully correct answer.** Do not reverse-
engineer your analysis to reproduce a quoted figure.

---
# Part 2 — Setup

Run these three cells before anything else.

In [ ]:
# --- Cell S1: dependencies -------------------------------------------------
!pip install -q kagglehub pandas matplotlib seaborn

import os, re, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 2.2.2 | numpy 2.0.2


In [ ]:
# --- Cell S2: download the data -------------------------------------------
# Kaggle needs credentials. In Colab the simplest route is kagglehub.login(),
# which prompts for your Kaggle username and API key.
#   Get a key at: https://www.kaggle.com/settings  ->  API  ->  "Create New Token"
#   That downloads kaggle.json, which contains the username and key.

import kagglehub

try:
    kagglehub.login()                       # interactive prompt
except Exception as e:
    print("Interactive login unavailable:", e)
    # Fallback: set credentials directly (do NOT commit real keys to a repo)
    # os.environ["KAGGLE_USERNAME"] = "your_username"
    # os.environ["KAGGLE_KEY"]      = "your_key"

DATA_DIR = kagglehub.dataset_download("ismatkhan121/uidai-data-hackathon-2026-dataset")
print("Downloaded to:", DATA_DIR)

Using Colab cache for faster access to the 'uidai-data-hackathon-2026-dataset' dataset.
Downloaded to: /kaggle/input/uidai-data-hackathon-2026-dataset
Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
# --- Cell S3: inspect the raw files ---------------------------------------
# Do NOT load them yet. Listing and understanding the file layout is Q1.

files = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
print(f"{len(files)} CSV files found\n")
for f in files:
    print(f"{os.path.basename(f):<55} {os.path.getsize(f)/1e6:8.2f} MB")

# Peek at one file's header and first rows, without reading the whole thing.
print("\n--- head of first file ---")
print(pd.read_csv(files[0], nrows=5))

12 CSV files found

api_data_aadhar_biometric_0_500000.csv                     22.33 MB
api_data_aadhar_biometric_1000000_1500000.csv              22.09 MB
api_data_aadhar_biometric_1500000_1861108.csv              15.93 MB
api_data_aadhar_biometric_500000_1000000.csv               22.09 MB
api_data_aadhar_demographic_0_500000.csv                   22.13 MB
api_data_aadhar_demographic_1000000_1500000.csv            22.12 MB
api_data_aadhar_demographic_1500000_2000000.csv            22.10 MB
api_data_aadhar_demographic_2000000_2071700.csv             3.18 MB
api_data_aadhar_demographic_500000_1000000.csv             22.08 MB
api_data_aadhar_enrolment_0_500000.csv                     22.81 MB
api_data_aadhar_enrolment_1000000_1006029.csv               0.28 MB
api_data_aadhar_enrolment_500000_1000000.csv               22.80 MB

--- head of first file ---
         date              state      district  pincode  bio_age_5_17  bio_age_17_
0  01-03-2025            Haryana  Mahendragarh   1230

In [ ]:
print(pd.read_csv(files[1], nrows=15))

          date             state      district  pincode  bio_age_5_17  bio_age_17_
0   07-11-2025           Haryana  Yamuna Nagar   135002             4            6
1   07-11-2025           Haryana   Yamunanagar   135001             1            2
2   07-11-2025  Himachal Pradesh      Bilaspur   174004             2            2
3   07-11-2025  Himachal Pradesh      Bilaspur   174005             1            0
4   07-11-2025  Himachal Pradesh      Bilaspur   174013             3            1
5   07-11-2025  Himachal Pradesh      Bilaspur   174021            21            4
6   07-11-2025  Himachal Pradesh      Bilaspur   174024             0            1
7   07-11-2025  Himachal Pradesh      Bilaspur   174027             2            3
8   07-11-2025  Himachal Pradesh      Bilaspur   174029            12            5
9   07-11-2025  Himachal Pradesh      Bilaspur   174031             1            1
10  07-11-2025  Himachal Pradesh      Bilaspur   174201             2            1
11  

In [ ]:
print(pd.read_csv(files[5], nrows=20))

          date       state    district  pincode  demo_age_5_17  demo_age_17_
0   01-11-2025  Puducherry    Karaikal   609603              0             4
1   01-11-2025  Puducherry    Karaikal   609605              0             3
2   01-11-2025  Puducherry    Karaikal   609606              0             5
3   01-11-2025  Puducherry  Puducherry   605011              2             6
4   01-11-2025  Puducherry  Puducherry   605111              0             1
5   01-11-2025  Puducherry  Puducherry   673310              1             5
6   01-11-2025  Puducherry       Yanam   533464              0             1
7   01-11-2025      Punjab    Amritsar   143001             22           217
8   01-11-2025      Punjab    Amritsar   143002              0             8
9   01-11-2025      Punjab    Amritsar   143022              0            16
10  01-11-2025      Punjab    Amritsar   143105              1            56
11  01-11-2025      Punjab    Amritsar   143107              0            11

In [ ]:
print(pd.read_csv(files[11], nrows=20))

          date           state                     district  pincode  age_0_5  age_5_17  age_18_greater
0   26-10-2025  Andhra Pradesh                     Nalgonda   508004        0         1               0
1   26-10-2025  Andhra Pradesh                     Nalgonda   508238        1         0               0
2   26-10-2025  Andhra Pradesh                     Nalgonda   508278        1         0               0
3   26-10-2025  Andhra Pradesh                      Nandyal   518432        0         1               0
4   26-10-2025  Andhra Pradesh                      Nandyal   518543        1         0               0
5   26-10-2025  Andhra Pradesh                      Nellore   524003        0         1               0
6   26-10-2025  Andhra Pradesh                      Nellore   524129        2         0               0
7   26-10-2025  Andhra Pradesh                      Nellore   524132        3         0               0
8   26-10-2025  Andhra Pradesh                      Nellore   52

## Section A — Assembly and Profiling

Answer all questions. This lab carries 100 marks (5 marks each, 20 questions) and the weight is 3 marks. Deadline: 5 PM

## Q1. Reassemble the three tables

The 12 files are split by row range, encoded in the filenames. Parse those filenames
**programmatically** to determine the intended ranges, and verify *before* concatenating that
the ranges are contiguous, non-overlapping, and complete.

Then confirm your assembled tables contain exactly **1,006,029**, **2,071,700** and
**1,861,108** rows. Report any discrepancy — do not silently accept a mismatch.

**Deliverable:** Three DataFrames named `enrol`, `demo`, `bio`, plus a printed verification of the row counts. **Questions 2-20 depends on these three names.**

In [ ]:
# Filenames look like: api_data_aadhar_<table>_<start>_<end>.csv
PATTERN = re.compile(r"api_data_aadhar_(?P<table>\w+?)_(?P<start>\d+)_(?P<end>\d+)\.csv$")

# TODO 1a: parse every filename into (table, start, end). Build a manifest DataFrame.

# TODO 1b: for each table, sort the parts by `start` and CHECK the ranges before loading:
#          - do the parts begin at 0?
#          - is every part's `start` equal to the previous part's `end`?  (no gaps)
#          - does any part overlap another?
#          Print a clear PASS/FAIL per table. Do not proceed silently on FAIL.

# TODO 1c: concatenate each table's parts in order.
enrol = None   # 1,006,029 rows expected
demo  = None   # 2,071,700 rows expected
bio   = None   # 1,861,108 rows expected

# TODO 1d: verify actual vs expected row counts and report any discrepancy.
EXPECTED = {"enrol": 1_006_029, "demo": 2_071_700, "bio": 1_861_108}

In [ ]:
PATTERN = re.compile(r"api_data_aadhar_(?P<table>\w+?)_(?P<start>\d+)_(?P<end>\d+)\.csv$")

manifest_data = []
for f_path in files:
    basename = os.path.basename(f_path)
    match = PATTERN.match(basename)
    if match:
        manifest_data.append(match.groupdict())
manifest = pd.DataFrame(manifest_data)
manifest['start'] = manifest['start'].astype(int)
manifest['end'] = manifest['end'].astype(int)
manifest['filepath'] = [f for f in files if PATTERN.match(os.path.basename(f))]
display(manifest.head())


enrol_files = manifest[manifest['table'] == 'enrolment'].sort_values('start')['filepath'].tolist()
enrol = pd.concat([pd.read_csv(f) for f in enrol_files], ignore_index=True)

demo_files = manifest[manifest['table'] == 'demographic'].sort_values('start')['filepath'].tolist()
demo = pd.concat([pd.read_csv(f) for f in demo_files], ignore_index=True)

bio_files = manifest[manifest['table'] == 'biometric'].sort_values('start')['filepath'].tolist()
bio = pd.concat([pd.read_csv(f) for f in bio_files], ignore_index=True)


,table,start,end,filepath
0,biometric,0,500000,/kaggle/input/uidai-data-hackathon-2026-datase...
1,biometric,1000000,1500000,/kaggle/input/uidai-data-hackathon-2026-datase...
2,biometric,1500000,1861108,/kaggle/input/uidai-data-hackathon-2026-datase...
3,biometric,500000,1000000,/kaggle/input/uidai-data-hackathon-2026-datase...
4,demographic,0,500000,/kaggle/input/uidai-data-hackathon-2026-datase...


In [ ]:
print(enrol_files)

['/kaggle/input/uidai-data-hackathon-2026-dataset/api_data_aadhar_enrolment/api_data_aadhar_enrolment/api_data_aadhar_enrolment_0_500000.csv', '/kaggle/input/uidai-data-hackathon-2026-dataset/api_data_aadhar_enrolment/api_data_aadhar_enrolment/api_data_aadhar_enrolment_500000_1000000.csv', '/kaggle/input/uidai-data-hackathon-2026-dataset/api_data_aadhar_enrolment/api_data_aadhar_enrolment/api_data_aadhar_enrolment_1000000_1006029.csv']


In [ ]:
enrol.describe()

,pincode,age_0_5,age_5_17,age_18_greater
count,1.006029e+06,1.006029e+06,1.006029e+06,1.006029e+06
mean,5.186415e+05,3.525709e+00,1.710074e+00,1.673441e-01
std,2.056360e+05,1.753851e+01,1.436963e+01,3.220525e+00
min,1.000000e+05,0.000000e+00,0.000000e+00,0.000000e+00
25%,3.636410e+05,1.000000e+00,0.000000e+00,0.000000e+00
50%,5.174170e+05,2.000000e+00,0.000000e+00,0.000000e+00
75%,7.001040e+05,3.000000e+00,1.000000e+00,0.000000e+00
max,8.554560e+05,2.688000e+03,1.812000e+03,8.550000e+02


In [ ]:
print(len(enrol))

1006029


In [ ]:
actual_counts = {
    'enrol': len(enrol),
    'demo': len(demo),
    'bio': len(bio)
}
for table_name, expected_count in EXPECTED.items():
    actual_count = actual_counts[table_name]
    if actual_count == expected_count:
        print(f'PASS: {table_name} - Actual: {actual_count}, Expected: {expected_count}')
    else:
        print(f'FAIL: {table_name} - Actual: {actual_count}, Expected: {expected_count} (Discrepancy: {actual_count - expected_count})')

PASS: enrol - Actual: 1006029, Expected: 1006029
PASS: demo - Actual: 2071700, Expected: 2071700
PASS: bio - Actual: 1861108, Expected: 1861108


In [ ]:
manifest.head(12)

,table,start,end,filepath
0,biometric,0,500000,/kaggle/input/uidai-data-hackathon-2026-datase...
1,biometric,1000000,1500000,/kaggle/input/uidai-data-hackathon-2026-datase...
2,biometric,1500000,1861108,/kaggle/input/uidai-data-hackathon-2026-datase...
3,biometric,500000,1000000,/kaggle/input/uidai-data-hackathon-2026-datase...
4,demographic,0,500000,/kaggle/input/uidai-data-hackathon-2026-datase...
5,demographic,1000000,1500000,/kaggle/input/uidai-data-hackathon-2026-datase...
6,demographic,1500000,2000000,/kaggle/input/uidai-data-hackathon-2026-datase...
7,demographic,2000000,2071700,/kaggle/input/uidai-data-hackathon-2026-datase...
8,demographic,500000,1000000,/kaggle/input/uidai-data-hackathon-2026-datase...
9,enrolment,0,500000,/kaggle/input/uidai-data-hackathon-2026-datase...


### ✍️ Your answer — Q1

_Replace this text with your written justification. Every judgement call (a threshold, a merge rule, a dropped row, a bin width) needs a sentence explaining why. Unjustified choices are marked as if the choice were wrong._

## Q2. Build the data dictionary and fix the dtypes

Document every column: semantic type, correct pandas dtype, and what it represents.

Then justify **three specific decisions**:
1. How you store `pincode` — and why the default `read_csv` behaviour is wrong.
2. How you parse `date`.
3. What dtype the age-band count columns should carry.

Report the memory saved by your corrections against a naive `read_csv`.

**Deliverable:** A data dictionary table, plus before/after `memory_usage(deep=True)` figures and the % saved.

In [ ]:
# TODO 2a: record the naive memory footprint of all three tables.
naive_mem = None   # bytes, using .memory_usage(deep=True).sum()
naive_mem_enrol = enrol.memory_usage(deep=True).sum()
naive_mem_demo = demo.memory_usage(deep=True).sum()
naive_mem_bio = bio.memory_usage(deep=True).sum()
naive_mem = naive_mem_enrol + naive_mem_demo + naive_mem_bio
print(f"Naive memory: {naive_mem / 1e6:.2f} MB")

# TODO 2b: write your data dictionary as a DataFrame with columns:
#          table | column | semantic_type | correct_dtype | description
data_dictionary_rows = [
    # Enrolment table
    {"table": "enrolment", "column": "date", "semantic_type": "temporal", "correct_dtype": "datetime64[ns]", "description": "Date of enrolment activity"},
    {"table": "enrolment", "column": "state", "semantic_type": "categorical", "correct_dtype": "category", "description": "State where enrolment occurred"},
    {"table": "enrolment", "column": "district", "semantic_type": "categorical", "correct_dtype": "category", "description": "District where enrolment occurred"},
    {"table": "enrolment", "column": "pincode", "semantic_type": "identifier", "correct_dtype": "string", "description": "Pincode of the enrolment location (kept as string to preserve leading zeros)"},
    {"table": "enrolment", "column": "age_0_5", "semantic_type": "numerical", "correct_dtype": "uint16", "description": "Number of enrolments for age group 0-5"},
    {"table": "enrolment", "column": "age_5_17", "semantic_type": "numerical", "correct_dtype": "uint16", "description": "Number of enrolments for age group 5-17"},
    {"table": "enrolment", "column": "age_18_greater", "semantic_type": "numerical", "correct_dtype": "uint16", "description": "Number of enrolments for age group 18 and greater"},

    # Demographic table
    {"table": "demographic", "column": "date", "semantic_type": "temporal", "correct_dtype": "datetime64[ns]", "description": "Date of demographic update activity"},
    {"table": "demographic", "column": "state", "semantic_type": "categorical", "correct_dtype": "category", "description": "State where demographic update occurred"},
    {"table": "demographic", "column": "district", "semantic_type": "categorical", "correct_dtype": "category", "description": "District where demographic update occurred"},
    {"table": "demographic", "column": "pincode", "semantic_type": "identifier", "correct_dtype": "string", "description": "Pincode of the demographic update location (kept as string to preserve leading zeros)"},
    {"table": "demographic", "column": "demo_age_5_17", "semantic_type": "numerical", "correct_dtype": "uint16", "description": "Number of demographic updates for age group 5-17"},
    {"table": "demographic", "column": "demo_age_17_", "semantic_type": "numerical", "correct_dtype": "uint16", "description": "Number of demographic updates for age group 17 and greater"},

    # Biometric table
    {"table": "biometric", "column": "date", "semantic_type": "temporal", "correct_dtype": "datetime64[ns]", "description": "Date of biometric update activity"},
    {"table": "biometric", "column": "state", "semantic_type": "categorical", "correct_dtype": "category", "description": "State where biometric update occurred"},
    {"table": "biometric", "column": "district", "semantic_type": "categorical", "correct_dtype": "category", "description": "District where biometric update occurred"},
    {"table": "biometric", "column": "pincode", "semantic_type": "identifier", "correct_dtype": "string", "description": "Pincode of the biometric update location (kept as string to preserve leading zeros)"},
    {"table": "biometric", "column": "bio_age_5_17", "semantic_type": "numerical", "correct_dtype": "uint16", "description": "Number of biometric updates for age group 5-17"},
    {"table": "biometric", "column": "bio_age_17_", "semantic_type": "numerical", "correct_dtype": "uint16", "description": "Number of biometric updates for age group 17 and greater"}
]
data_dictionary = pd.DataFrame(data_dictionary_rows)
# display(data_dictionary)

# TODO 2c: apply the corrected dtypes.
#          Think hard about `pincode` before you write anything.
#          Hint: what happens to a value like 001234 when pandas reads it as int64?
#          Hint: `state` and `district` are low-cardinality -> consider 'category'.
enrol['date'] = pd.to_datetime(enrol['date'], format='%d-%m-%Y', errors='coerce')
enrol['pincode'] = enrol['pincode'].astype(str).str.zfill(6) # Ensure 6 digits, pad with leading zeros
enrol['state'] = enrol['state'].astype('category')
enrol['district'] = enrol['district'].astype('category')
enrol['age_0_5'] = enrol['age_0_5'].astype('uint16')
enrol['age_5_17'] = enrol['age_5_17'].astype('uint16')
enrol['age_18_greater'] = enrol['age_18_greater'].astype('uint16')

# Apply dtypes to demo
demo['date'] = pd.to_datetime(demo['date'], format='%d-%m-%Y', errors='coerce')
demo['pincode'] = demo['pincode'].astype(str).str.zfill(6)
demo['state'] = demo['state'].astype('category')
demo['district'] = demo['district'].astype('category')
demo['demo_age_5_17'] = demo['demo_age_5_17'].astype('uint16')
demo['demo_age_17_'] = demo['demo_age_17_'].astype('uint16')

# Apply dtypes to bio
bio['date'] = pd.to_datetime(bio['date'], format='%d-%m-%Y', errors='coerce')
bio['pincode'] = bio['pincode'].astype(str).str.zfill(6)
bio['state'] = bio['state'].astype('category')
bio['district'] = bio['district'].astype('category')
bio['bio_age_5_17'] = bio['bio_age_5_17'].astype('uint16')
bio['bio_age_17_'] = bio['bio_age_17_'].astype('uint16')

print('Dtype Corrections')
print('\nEnrol Dtypes:')
display(enrol.info(verbose=False, memory_usage="deep"))
print('\nDemo Dtypes:')
display(demo.info(verbose=False, memory_usage="deep"))
print('\nBio Dtypes:')
display(bio.info(verbose=False, memory_usage="deep"))


# TODO 2d: report optimised memory and the percentage saved.
optimised_mem_enrol = enrol.memory_usage(deep=True).sum()
optimised_mem_demo = demo.memory_usage(deep=True).sum()
optimised_mem_bio = bio.memory_usage(deep=True).sum()
optimised_mem = optimised_mem_enrol + optimised_mem_demo + optimised_mem_bio

print(f"Optimised memory: {optimised_mem / 1e6:.2f} MB")
print(f"Memory saved: {100 * (naive_mem - optimised_mem) / naive_mem:.2f}%")

Naive memory: 994.44 MB
Dtype Corrections

Enrol Dtypes:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1006029 entries, 0 to 1006028
Columns: 7 entries, date to age_18_greater
dtypes: category(2), datetime64[ns](1), object(1), uint16(3)
memory usage: 69.2 MB


None


Demo Dtypes:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2071700 entries, 0 to 2071699
Columns: 6 entries, date to demo_age_17_
dtypes: category(2), datetime64[ns](1), object(1), uint16(2)
memory usage: 138.4 MB


None


Bio Dtypes:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1861108 entries, 0 to 1861107
Columns: 6 entries, date to bio_age_17_
dtypes: category(2), datetime64[ns](1), object(1), uint16(2)
memory usage: 124.3 MB


None

Optimised memory: 348.02 MB
Memory saved: 65.00%


### ✍️ Your answer — Q2

_Replace this text with your written justification. Every judgement call (a threshold, a merge rule, a dropped row, a bin width) needs a sentence explaining why. Unjustified choices are marked as if the choice were wrong._

## Q3. Profile all three tables

For every column in every table, compute: row count, null count, distinct count, and — for the
count columns — min, max, mean, median and standard deviation.

Present it as **one comparison table**. Flag anything impossible on sight: negative counts,
absurd maxima, nulls where the schema implies there should be none.

**Deliverable:** A single tidy profile DataFrame covering all three tables, plus a short list of red flags.

In [ ]:
def profile(df: pd.DataFrame, table_name: str) -> pd.DataFrame:
    """Return one row per column: nulls, distinct count, and numeric summary stats."""
    # TODO 3a: implement. Return a tidy frame, one row per column.
    profile_data = []
    for col in df.columns:
        col_series = df[col]
        row_count = len(col_series)
        null_count = col_series.isnull().sum()
        distinct_count = col_series.nunique()

        stats = {
            'table': table_name,
            'column': col,
            'row_count': row_count,
            'null_count': null_count,
            'distinct_count': distinct_count,
        }

        if df.dtypes[col] in ['int64', 'uint16']:
              stats['min'] = col_series.min()
              stats['max'] = col_series.max()
              stats['mean'] = col_series.mean()
              stats['median'] = col_series.median()
              stats['std'] = col_series.std()
        else:
              stats['min'] = None
              stats['max'] = None
              stats['mean'] = None
              stats['median'] = None
              stats['std'] = None
        profile_data.append(stats)
    # raise NotImplementedError
    return pd.DataFrame(profile_data)


# TODO 3b: build the combined profile across enrol / demo / bio and display it.
enrol_profile = profile(enrol, 'enrolment')
demo_profile = profile(demo, 'demographic')
bio_profile = profile(bio, 'biometric')

combined_profile = pd.concat([enrol_profile, demo_profile, bio_profile], ignore_index=True)
display(combined_profile)


# TODO 3c: list every value you consider impossible or suspicious, with the evidence.
print("1. Null counts for 'date', 'state', 'district', 'pincode' should ideally be zero after Q2. Any non-zero count here might indicate issues with `pd.to_datetime` or `astype(str).str.zfill(6)` in Q2.")
print("2. `pincode` min/max: After `str.zfill(6)`, the 'pincode' column is a string. Its min/max as a string might not be numerically meaningful, but its distinct count is important.")
print("3. Age-band counts (e.g., 'age_0_5', 'bio_age_5_17'): If any of these columns show negative minimum values, that would be a critical data error, as counts cannot be negative. (Based on `describe()` output from `enrol`, they are all >= 0, which is good.)")
print("4. Distinct counts for 'state' and 'district' columns are expected to be relatively low compared to `row_count`, especially after canonicalization, indicating their categorical nature. High distinct counts (close to `row_count`) could suggest issues in data entry or lack of canonicalization.")
print("5. Any `null_count` for age-band columns would indicate missing counts, which might need imputation or specific handling depending on the context.")
print("6.There is a state named 10000")

,table,column,row_count,null_count,distinct_count,min,max,mean,median,std
0,enrolment,date,1006029,0,92,NaN,NaN,NaN,NaN,NaN
1,enrolment,state,1006029,0,55,NaN,NaN,NaN,NaN,NaN
2,enrolment,district,1006029,0,985,NaN,NaN,NaN,NaN,NaN
3,enrolment,pincode,1006029,0,19463,NaN,NaN,NaN,NaN,NaN
4,enrolment,age_0_5,1006029,0,671,0.0,2688.0,3.525709,2.0,17.538511
5,enrolment,age_5_17,1006029,0,624,0.0,1812.0,1.710074,0.0,14.369625
6,enrolment,age_18_greater,1006029,0,199,0.0,855.0,0.167344,0.0,3.220525
7,demographic,date,2071700,0,95,NaN,NaN,NaN,NaN,NaN
8,demographic,state,2071700,0,65,NaN,NaN,NaN,NaN,NaN
9,demographic,district,2071700,0,983,NaN,NaN,NaN,NaN,NaN


1. Null counts for 'date', 'state', 'district', 'pincode' should ideally be zero after Q2. Any non-zero count here might indicate issues with `pd.to_datetime` or `astype(str).str.zfill(6)` in Q2.
2. `pincode` min/max: After `str.zfill(6)`, the 'pincode' column is a string. Its min/max as a string might not be numerically meaningful, but its distinct count is important.
3. Age-band counts (e.g., 'age_0_5', 'bio_age_5_17'): If any of these columns show negative minimum values, that would be a critical data error, as counts cannot be negative. (Based on `describe()` output from `enrol`, they are all >= 0, which is good.)
4. Distinct counts for 'state' and 'district' columns are expected to be relatively low compared to `row_count`, especially after canonicalization, indicating their categorical nature. High distinct counts (close to `row_count`) could suggest issues in data entry or lack of canonicalization.
5. Any `null_count` for age-band columns would indicate missing counts, which mig

### ✍️ Your answer — Q3

_Replace this text with your written justification. Every judgement call (a threshold, a merge rule, a dropped row, a bin width) needs a sentence explaining why. Unjustified choices are marked as if the choice were wrong._

## Q4. Test the grain

Is `(date, state, district, pincode)` a unique key in each table?

Quantify **three separate cases**:
- fully identical duplicate rows,
- rows with the same key but *different* counts,
- keys appearing more than twice.

For the second case, decide whether to **sum**, take the **maximum**, or **drop** — and defend
it, because the three give materially different totals.

One participant team reported removing up to **22.9%** duplicates. Check whether you can
reproduce a figure of that magnitude.

**Deliverable:** Counts for all three cases per table, your chosen resolution rule, and the national total under each of the three candidate rules so the cost of the choice is visible.

In [ ]:
KEY = ["date", "state", "district", "pincode"]

# TODO 4a: count fully identical duplicate rows in each table.

identical_duplicates_enrol = enrol.duplicated().sum()
print(f"Enrolment table has {identical_duplicates_enrol} identical duplicate rows.")
identical_duplicates_demo = demo.duplicated().sum()
print(f"Demographic table has {identical_duplicates_demo} identical duplicate rows.")
identical_duplicates_bio = bio.duplicated().sum()
print(f"Biometric table has {identical_duplicates_bio} identical duplicate rows.")

# TODO 4b: count key collisions where the non-key values DIFFER.

key_duplicates_mask = enrol.duplicated(subset=KEY, keep=False)
key_duplicate_rows = enrol[key_duplicates_mask].sort_values(by=KEY)
print(f"Enrolment table has {len(key_duplicate_rows)} key collisions where the non-key values differ.")
key_duplicate_mask_demo = demo.duplicated(subset=KEY, keep=False)
key_duplicate_rows_demo = demo[key_duplicate_mask_demo].sort_values(by=KEY)
print(f"Demographic table has {len(key_duplicate_rows_demo)} key collisions where the non-key values differ.")
key_duplicate_mask_bio = bio.duplicated(subset=KEY, keep=False)
key_duplicate_rows_bio = bio[key_duplicate_mask_bio].sort_values(by=KEY)
print(f"Biometric table has {len(key_duplicate_rows_bio)} key collisions where the non-key values differ.")

# TODO 4c: distribution of key multiplicity (how many keys appear 2x, 3x, ...).

# TODO 4d: compute the national total under each rule: sum / max / drop-all-dupes.
#          Show the three numbers side by side. Then pick one and justify it.

# TODO 4e: can you reproduce a ~22.9% duplicate rate? Under what definition of "duplicate"?
#          If not, say so plainly and give your figure instead.

Enrolment table has 22957 identical duplicate rows.
Demographic table has 473601 identical duplicate rows.
Biometric table has 94896 identical duplicate rows.
Enrolment table has 45914 key collisions where the non-key values differ.
Demographic table has 947202 key collisions where the non-key values differ.
Biometric table has 189792 key collisions where the non-key values differ.


### ✍️ Your answer — Q4

_Replace this text with your written justification. Every judgement call (a threshold, a merge rule, a dropped row, a bin width) needs a sentence explaining why. Unjustified choices are marked as if the choice were wrong._

## Section B — Reference Data Integrity

## Q5. Reconcile the states against ground truth

India has **28 states and 8 union territories** — 36 in total. Count the distinct `state`
values in each table and explain the excess.

Build a canonical mapping from every raw value to its official entity, and report how many raw
values collapsed into each canonical one. Identify any raw value you could **not** confidently
map, and say what you would do with it in production.

**Deliverable:** A reusable `state_map` dictionary or lookup DataFrame (this is a graded artifact — keep it separate and readable), plus a table of raw→canonical collapse counts.

In [ ]:
# The 36 official entities. Use this as ground truth.
OFFICIAL_STATES = [
    "Andhra Pradesh","Arunachal Pradesh","Assam","Bihar","Chhattisgarh","Goa","Gujarat",
    "Haryana","Himachal Pradesh","Jharkhand","Karnataka","Kerala","Madhya Pradesh",
    "Maharashtra","Manipur","Meghalaya","Mizoram","Nagaland","Odisha","Punjab","Rajasthan",
    "Sikkim","Tamil Nadu","Telangana","Tripura","Uttar Pradesh","Uttarakhand","West Bengal",
    "Andaman and Nicobar Islands","Chandigarh",
    "Dadra and Nagar Haveli and Daman and Diu","Delhi","Jammu and Kashmir","Ladakh",
    "Lakshadweep","Puducherry",
]
assert len(OFFICIAL_STATES) == 36

# TODO 5a: distinct raw state values per table, and the union across all three.

enrol_distinct_state_counts = enrol['state'].nunique()
demo_distinct_state_counts = demo['state'].nunique()
bio_distinct_state_counts = bio['state'].nunique()

print(f"Distinct state values in enrolment table: {enrol_distinct_state_counts}")
print(f"Distinct state values in demographic table: {demo_distinct_state_counts}")
print(f"Distinct state values in biometric table: {bio_distinct_state_counts}")
union_states_list = list(set(enrol['state'].unique()) | set(demo['state'].unique()) | set(bio['state'].unique()))
union_states_count = len(union_states_list)
print(union_states_list)
print(union_states_count)

# TODO 5b: normalise progressively and measure the effect of EACH step separately:
#          strip whitespace -> case-fold -> collapse internal spaces -> punctuation ->
#          known aliases (e.g. "Orissa"/"Odisha", "Pondicherry"/"Puducherry").
#          Report how many distinct values survive after each step.

import re

# Create a set of canonical official state names for filtering (lower-cased, single spaces, stripped)
official_states_canonical_set = {re.sub(r'\s+', ' ', s.strip().lower().replace('&', 'and')) for s in OFFICIAL_STATES}

print("Progressive Normalization of State Names")

current_states_list_for_progress = list(union_states_list)
print(f"Initial distinct values (from union_states_list): {len(set(current_states_list_for_progress))}")

current_states_list_for_progress = [s.strip() for s in current_states_list_for_progress]
print(f"After stripping whitespace: {len(set(current_states_list_for_progress))} distinct values")

current_states_list_for_progress = [s.lower() for s in current_states_list_for_progress]
print(f"After case-folding: {len(set(current_states_list_for_progress))} distinct values")

current_states_list_for_progress = [re.sub(r'\s+', ' ', s) for s in current_states_list_for_progress]
print(f"After collapsing internal spaces: {len(set(current_states_list_for_progress))} distinct values")

current_states_list_for_progress = [s.replace('&', 'and') for s in current_states_list_for_progress]
print(f"After replacing '&' with 'and': {len(set(current_states_list_for_progress))} distinct values")

alias_mapping = {
    "orissa": "odisha",
    "pondicherry": "puducherry",
    "the dadra and nagar haveli and daman and diu": "dadra and nagar haveli and daman and diu",
    "daman and diu": "dadra and nagar haveli and daman and diu",
    "dadra and nagar haveli": "dadra and nagar haveli and daman and diu",
    "westbengal": "west bengal",
    "west bangal": "west bengal",
    "west bengli": "west bengal",
    "tamilnadu": "tamil nadu",
    "uttaranchal": "uttarakhand",
    "chhatisgarh": "chhattisgarh",
    "jammu and kashmir": "jammu and kashmir",
    "andaman and nicobar islands": "andaman and nicobar islands",
    "andhra pradesh": "andhra pradesh",
}

aliased_states_list = []
for state in current_states_list_for_progress:
    mapped_state = alias_mapping.get(state, state)
    # Broad patterns (e.g., if 'west bengal' is a substring, map it)
    if "west bengal" in mapped_state:
        mapped_state = "west bengal"
    elif "jammu and kashmir" in mapped_state:
        mapped_state = "jammu and kashmir"
    elif "andaman and nicobar islands" in mapped_state:
        mapped_state = "andaman and nicobar islands"
    elif "andhra pradesh" in mapped_state:
        mapped_state = "andhra pradesh"

    aliased_states_list.append(mapped_state)

current_states_list_for_progress = aliased_states_list
print(f"After applying known aliases: {len(set(current_states_list_for_progress))} distinct values")

# Step 6: Filter out non-state entries by checking against the official_states_canonical_set
final_cleaned_states = [s for s in current_states_list_for_progress if s in official_states_canonical_set]
print(f"After filtering for official states: {len(set(final_cleaned_states))} distinct values")

print("Unmapped Raw Values ")
unmapped_raw_values_set = set()
# Re-apply the full normalization pipeline for each raw state to determine final mapping
for raw_state in union_states_list:
    processed_state = raw_state.strip().lower().replace('&', 'and')
    processed_state = re.sub(r'\s+', ' ', processed_state)

    mapped_state = alias_mapping.get(processed_state, processed_state)
    if "west bengal" in mapped_state: mapped_state = "west bengal"
    elif "jammu and kashmir" in mapped_state: mapped_state = "jammu and kashmir"
    elif "andaman and nicobar islands" in mapped_state: mapped_state = "andaman and nicobar islands"
    elif "andhra pradesh" in mapped_state: mapped_state = "andhra pradesh"

    if mapped_state not in official_states_canonical_set:
        unmapped_raw_values_set.add(raw_state)

if unmapped_raw_values_set:
    print("List of raw values that could not be mapped to an official state:")
    for uv in sorted(list(unmapped_raw_values_set)): # Print unique unmapped values
        print(f"- {uv}")
    print("\nRecommended treatment: These values likely represent district names, city names, or erroneous entries. They should be investigated further or excluded from state-level analysis.")
else:
    print("All raw values from the union_states_list were successfully mapped to an official state.")



# TODO 5c: build the final mapping.
state_map = {}   # raw value -> canonical OFFICIAL_STATES entry

# TODO 5d: list every raw value you could NOT map, with your recommended treatment.
unmapped = []

Distinct state values in enrolment table: 55
Distinct state values in demographic table: 65
Distinct state values in biometric table: 57
['Jammu And Kashmir', 'Assam', 'The Dadra And Nagar Haveli And Daman And Diu', 'Uttar Pradesh', 'Pondicherry', 'Manipur', 'Andaman and Nicobar Islands', 'Chandigarh', 'Goa', 'Mizoram', 'Daman & Diu', 'Orissa', 'WESTBENGAL', 'West Bangal', 'Daman and Diu', 'Punjab', 'Dadra and Nagar Haveli', 'Puducherry', 'Telangana', '100000', 'Madhya Pradesh', 'Bihar', 'Lakshadweep', 'Dadra & Nagar Haveli', 'ODISHA', 'Nagpur', 'Maharashtra', 'Arunachal Pradesh', 'Darbhanga', 'BALANAGAR', 'Tamil Nadu', 'WEST BENGAL', 'Gujarat', 'Westbengal', 'Tripura', 'West bengal', 'West Bengal', 'Rajasthan', 'Andhra Pradesh', 'West Bengli', 'Sikkim', 'odisha', 'Madanapalle', 'Uttaranchal', 'west Bengal', 'andhra pradesh', 'Karnataka', 'Jammu & Kashmir', 'West  Bengal', 'Chhattisgarh', 'Chhatisgarh', 'Andaman & Nicobar Islands', 'Delhi', 'Odisha', 'Nagaland', 'Uttarakhand', 'Haryana

### ✍️ Your answer — Q5

_Replace this text with your written justification. Every judgement call (a threshold, a merge rule, a dropped row, a bin width) needs a sentence explaining why. Unjustified choices are marked as if the choice were wrong._

## Q6. Reconcile the districts

Repeat the exercise for `district`, against the official count of roughly **780–800**.

Then test a **separate** question: is `district` unique on its own, or does the same district
name appear in more than one state? Find the collisions.

State the consequence for anyone who groups by `district` without also grouping by `state`.

**Deliverable:** Distinct district counts before and after cleaning, a table of ambiguous district names with the states they appear in, and a worked example showing how much a district-only groupby distorts a real number.

In [ ]:
# TODO 6a: distinct district values before and after applying the same normalisation
#          strategy you built in Q5.
enrol_distinct_district_counts = enrol['district'].nunique()
demo_distinct_district_counts = demo['district'].nunique()
bio_distinct_district_counts = bio['district'].nunique()

print(f"Distinct district values in enrolment table: {enrol_distinct_district_counts}")
print(f"Distinct district values in demographic table: {demo_distinct_district_counts}")
print(f"Distinct district values in biometric table: {bio_distinct_district_counts}")

# TODO 6b: find district names that appear in more than one state.
#          (Real examples to sanity-check against: Aurangabad, Bilaspur, Hamirpur,
#           Balrampur, Pratapgarh. These are genuine collisions, not errors.)
state_counts_enrol = enrol.groupby("district")["state"].nunique()
state_counts_demo = demo.groupby("district")["state"].nunique()
state_counts_bio = bio.groupby("district")["state"].nunique()
print("Enrolment table district with repeated states:")
print(state_counts_enrol[state_counts_enrol > 1])
print("Demographic table district with repeated states:")
print(state_counts_demo[state_counts_demo > 1])
print("Biometric table district with repeated states:")
print(state_counts_bio[state_counts_bio > 1])

# TODO 6c: quantify the damage. Pick the most collided district name, compute its total
#          enrolments grouped by district ALONE vs grouped by (state, district).
#          Show the two numbers.

most_collided_district_name_enrol = state_counts_enrol[state_counts_enrol > 1].index[0]
most_collided_district_name_demo = state_counts_demo[state_counts_demo > 1].index[0]
most_collided_district_name_bio = state_counts_bio[state_counts_bio > 1].index[0]
print(f"Most collided district name in enrolment table: {most_collided_district_name_enrol}")
print(f"Most collided district name in demographic table: {most_collided_district_name_demo}")
print(f"Most collided district name in biometric table: {most_collided_district_name_bio}")






Distinct district values in enrolment table: 985
Distinct district values in demographic table: 983
Distinct district values in biometric table: 974
Enrolment table district with repeated states:
district
Adilabad      2
Anantnag      2
Anugul        2
Aurangabad    2
Badgam        2
             ..
Sundergarh    2
Udhampur      2
Warangal      2
Yanam         2
hooghly       2
Name: state, Length: 78, dtype: int64
Demographic table district with repeated states:
district
Adilabad         2
Anantnag         2
Angul            2
Anugul           2
Aurangabad       2
                ..
Tehri Garhwal    2
Udhampur         2
Warangal         2
Yanam            2
hooghly          3
Name: state, Length: 82, dtype: int64
Biometric table district with repeated states:
district
Adilabad      2
Almora        2
Anantnag      2
Angul         2
Anugul        2
             ..
Udhampur      2
Viluppuram    2
Warangal      2
Yanam         2
hooghly       3
Name: state, Length: 86, dtype: int64
Most c

### ✍️ Your answer — Q6

_Replace this text with your written justification. Every judgement call (a threshold, a merge rule, a dropped row, a bin width) needs a sentence explaining why. Unjustified choices are marked as if the choice were wrong._

## Q7. Validate pincodes and test the hierarchy

Check every pincode for structural validity — six digits, plausible leading digit, no values
mangled by integer conversion.

Then test whether the apparent `state → district → pincode` hierarchy actually holds: does each
pincode map to exactly one district, and each district to exactly one state? Quantify the
violations.

**Before concluding they are all errors**, consider whether some are genuine features of how
Indian postal geography works.

**Deliverable:** A validation report (rule, violation count, examples) and a reasoned verdict on whether the hierarchy violations are data errors, real-world geography, or both.

In [ ]:
# Indian PINs are 6 digits; the first digit encodes a postal zone and runs 1-8.
# Digit 0 and 9 are not valid leading digits.

# TODO 7a: structural validation. Count violations of each rule separately:
#          - not exactly 6 characters
#          - non-numeric characters
#          - leading digit outside 1-8
#          Did any of these get created by your own dtype handling in Q2? Check.
pincode_enrol = enrol['pincode']
pincode_demo = demo['pincode']
pincode_bio = bio['pincode']

non_numeric_enrol = pincode_enrol.str.contains(r'\D')
non_numeric_demo = pincode_demo.str.contains(r'\D')
non_numeric_bio = pincode_bio.str.contains(r'\D')

not_six_digits_enrol = pincode_enrol.str.len() != 6
not_six_digits_demo = pincode_demo.str.len() != 6
not_six_digits_bio = pincode_bio.str.len() != 6

invalid_leading_digit_enrol = pincode_enrol.str[0].isin(['0', '9'])
invalid_leading_digit_demo = pincode_demo.str[0].isin(['0', '9'])
invalid_leading_digit_bio = pincode_bio.str[0].isin(['0', '9'])

print(f"Enrolment table has {non_numeric_enrol.sum()} non-numeric pincodes")
print(f"Demographic table has {non_numeric_demo.sum()} non-numeric pincodes")
print(f"Biometric table has {non_numeric_bio.sum()} non-numeric pincodes")

print(f"Enrolment table has {not_six_digits_enrol.sum()} pincodes with less than 6 digits")
print(f"Demographic table has {not_six_digits_demo.sum()} pincodes with less than 6 digits")
print(f"Biometric table has {not_six_digits_bio.sum()} pincodes with less than 6 digits")

print(f"Enrolment table has {invalid_leading_digit_enrol.sum()} pincodes with an invalid leading digit")
print(f"Demographic table has {invalid_leading_digit_demo.sum()} pincodes with an invalid leading digit")
print(f"Biometric table has {invalid_leading_digit_bio.sum()} pincodes with an invalid leading digit")
# TODO 7b: pincode -> district cardinality. How many pincodes map to >1 district?

# TODO 7c: district -> state cardinality. How many districts map to >1 state?
#          Cross-reference with your Q6 collision list before calling these errors.

# TODO 7d: write your verdict. Real Indian pincodes DO cross district boundaries.
#          Which of your violations are genuine geography and which are dirty data?

Enrolment table has 0 non-numeric pincodes
Demographic table has 0 non-numeric pincodes
Biometric table has 0 non-numeric pincodes
Enrolment table has 0 pincodes with less than 6 digits
Demographic table has 0 pincodes with less than 6 digits
Biometric table has 0 pincodes with less than 6 digits
Enrolment table has 0 pincodes with an invalid leading digit
Demographic table has 0 pincodes with an invalid leading digit
Biometric table has 0 pincodes with an invalid leading digit


### ✍️ Your answer — Q7

_Replace this text with your written justification. Every judgement call (a threshold, a merge rule, a dropped row, a bin width) needs a sentence explaining why. Unjustified choices are marked as if the choice were wrong._

## Q8. Reconcile geography across the three tables

Compare the sets of `(state, district, pincode)` combinations appearing in enrolment,
demographic updates and biometric updates. Which appear in all three, which in only one?

Do this **after** applying your canonicalisation from Q5–Q6, and report how much the overlap
improved as a result.

Interpret a pincode that appears in updates but never in enrolment — is that possible in
reality?

**Deliverable:** Overlap counts before and after canonicalisation (a 3-set comparison), plus your interpretation of each exclusive region.

In [ ]:
# TODO 8a: build the location set for each table (before canonicalisation).

# TODO 8b: compute all 7 regions of the 3-way overlap. A Venn diagram is optional;
#          the numbers are not.

# TODO 8c: repeat AFTER canonicalisation. Report the improvement.
#          If canonicalisation did NOT improve overlap, that is an interesting result --
#          say so and explain what it implies.

# TODO 8d: interpret. Can a location have biometric updates but zero enrolments, ever?
#          What does it mean if it does?

### ✍️ Your answer — Q8

_Replace this text with your written justification. Every judgement call (a threshold, a merge rule, a dropped row, a bin width) needs a sentence explaining why. Unjustified choices are marked as if the choice were wrong._

## Section C — Time and Missingness

## Q9. Audit the date coverage

Parse `date` and establish the true range in each table. Are all dates present, or are there
gaps? Test for weekly structure — do weekends differ from weekdays? Check whether coverage
begins at the same time in every state.

Report whether the **March–December 2025** range quoted from participant analyses is accurate.

**Deliverable:** Actual min/max dates per table, a list of missing dates, a weekday effect test, and a verdict on the quoted date range.

In [ ]:
# TODO 9a: parse dates robustly. Check for and report any unparseable values BEFORE
#          coercing them to NaT -- silent coercion hides real problems.

# TODO 9b: actual date range per table. Does it match the claimed Mar-Dec 2025?

# TODO 9c: build the full calendar between min and max. Which dates have zero rows?

# TODO 9d: weekday effect -- mean daily total by day of week. Is the difference large
#          enough to matter for any rolling average you might compute later?

# TODO 9e: per-state first-reporting-date. Do all states start together?

### ✍️ Your answer — Q9

_Replace this text with your written justification. Every judgement call (a threshold, a merge rule, a dropped row, a bin width) needs a sentence explaining why. Unjustified choices are marked as if the choice were wrong._

## Q10. Decide what absence means

Construct the complete grid of (date × pincode) that would exist if every location reported
every day, and compare it to the rows actually present. Quantify the sparsity.

Then argue, **with evidence from the data**, whether a missing row means a true zero or an
unreported day — and show how *mean daily enrolment per pincode* changes depending on which
interpretation you adopt.

This is the most consequential decision in the assignment. Treat it as such.

**Deliverable:** The sparsity figure, both versions of the mean, and a defended choice between them.

In [ ]:
# TODO 10a: size of the complete grid = n_distinct_dates * n_distinct_pincodes.
#           Compare to actual row count. Report the fill rate.

# TODO 10b: compute mean daily enrolment per pincode TWO ways:
#           (i)  dense  -- absent rows treated as 0
#           (ii) sparse -- absent rows excluded from the denominator
#           Show both numbers. The gap between them is the size of the decision.

# TODO 10c: find EVIDENCE that distinguishes the two interpretations. Ideas:
#           - do explicit zero-valued rows exist in the data? if the file records
#             explicit zeros somewhere, absence probably means "not reported"
#           - do high-volume urban pincodes have gaps? unlikely to be true zeros
#           - are gaps clustered by date (outage) or by location (never reports)?

# TODO 10d: state your choice and carry it consistently through Q11-Q20.

### ✍️ Your answer — Q10

_Replace this text with your written justification. Every judgement call (a threshold, a merge rule, a dropped row, a bin width) needs a sentence explaining why. Unjustified choices are marked as if the choice were wrong._

## Q11. Investigate the 215% growth claim

A participant team reported an **average monthly growth rate of 215%** in enrolments.
Reproduce their calculation as best you can.

Then test the alternative explanation: that early months have thinner reporting coverage, so
the apparent growth is an artifact of data collection ramping up rather than real change in
enrolment behaviour.

Distinguish the two using **coverage metrics** — reporting pincodes per month, reporting days
per month — not just the totals.

**Deliverable:** The reproduced growth figure, coverage-adjusted figures beside it, and a defensible one-paragraph statement of what actually happened.

In [ ]:
# TODO 11a: monthly total enrolments. Compute month-over-month growth and its mean.
#           Can you get anywhere near 215%? Show your working.

# TODO 11b: coverage metrics per month:
#           - distinct reporting pincodes
#           - distinct reporting dates
#           - rows per reporting pincode
#           Plot or tabulate these ALONGSIDE the totals.

# TODO 11c: coverage-adjusted growth -- enrolments per reporting-pincode-day.
#           Does the growth survive the adjustment?

# TODO 11d: a fairer comparison -- restrict to the set of pincodes that report in
#           EVERY month (a balanced panel) and recompute growth on that subset.

# TODO 11e: your verdict. Real growth, reporting artifact, or a mix? Quantify the mix.

### ✍️ Your answer — Q11

_Replace this text with your written justification. Every judgement call (a threshold, a merge rule, a dropped row, a bin width) needs a sentence explaining why. Unjustified choices are marked as if the choice were wrong._

## Section D — Reshaping and Integration

## Q12. Make the data tidy

Reshape all three tables from wide to long, so that age band becomes a **value in a column**
rather than a set of column names.

Design the target schema first and **write it down before you code**. Verify that no counts
were lost or duplicated in the reshape, and state the memory and query-speed consequences of
long versus wide for this data.

**Deliverable:** Three long-format DataFrames, a conservation check (totals before == totals after), and a note on the memory/speed trade-off with measured numbers.

In [ ]:
# Target schema (write it here BEFORE coding):
#   date | state | district | pincode | event_type | age_band | count
#   where event_type in {enrolment, demographic_update, biometric_update}

# TODO 12a: melt each table. Keep the age band label meaningful, not the raw column name.

# TODO 12b: CONSERVATION CHECK -- the grand total of `count` must exactly equal the sum
#           of the original wide count columns. Assert it.

# TODO 12c: measure memory long vs wide, and time a groupby on each. Report both.

### ✍️ Your answer — Q12

_Replace this text with your written justification. Every judgement call (a threshold, a merge rule, a dropped row, a bin width) needs a sentence explaining why. Unjustified choices are marked as if the choice were wrong._

## Q13. Reconcile the age bands

Enrolment uses `0_5` / `5_17` / `18_greater`; updates use `5_17` / `17_plus`. **The boundary
between the top two bands is not the same in both.**

Determine — or, where the data cannot tell you, state clearly that it cannot — whether the
bands overlap, leave a gap, or are simply labelled loosely.

Propose a common age scheme allowing cross-table comparison, quantify the population you lose
or double-count under your scheme, and name one analysis your reconciliation would invalidate.

**Deliverable:** A documented, reusable reconciliation function (graded artifact), the size of the loss/double-count, and one named analysis your scheme breaks.

In [ ]:
# Enrolment: age_0_5, age_5_17, age_18_greater   -> boundary at 18
# Updates:   *_age_5_17, *_age_17_plus           -> boundary at 17
#
# Where does a 17-year-old go? In enrolment they fall in 5_17. In updates, 5_17 AND
# 17_plus both plausibly claim them. The column names alone cannot settle this.

# TODO 13a: state the three possible readings (overlap / gap / loose labelling) and
#           what each implies for the totals.

# TODO 13b: can the DATA settle it? Try: is (5_17 + 17_plus) internally consistent
#           across tables at the same location-date? Is there any location where the
#           bands sum to something checkable? Report honestly if it is undecidable.

def reconcile_age_bands(df: pd.DataFrame) -> pd.DataFrame:
    """Map each table's native age bands onto a common scheme.

    Document the scheme and its assumptions in this docstring.
    """
    # TODO 13c: implement.
    raise NotImplementedError

# TODO 13d: quantify -- how many events fall into the ambiguous zone, as a count and
#           as a % of the total? Name one analysis your scheme invalidates.

### ✍️ Your answer — Q13

_Replace this text with your written justification. Every judgement call (a threshold, a merge rule, a dropped row, a bin width) needs a sentence explaining why. Unjustified choices are marked as if the choice were wrong._

## Q14. Build the unified analysis table

Join the three tidied tables into a single table at a grain you choose and justify.

**Before running the join, predict its row count and write the prediction down.** Afterwards,
compare against the actual and explain any gap.

Justify your join type explicitly: an inner join here silently discards locations that enrol
but never update, which may be exactly the policy signal of interest.

**Deliverable:** A written prediction, the joined table, the predicted-vs-actual comparison, and an explicit defence of your join type.

In [ ]:
# TODO 14a: WRITE YOUR PREDICTION HERE BEFORE RUNNING THE JOIN.
predicted_rows = None
prediction_reasoning = """
(explain how you arrived at the number)
"""

# TODO 14b: choose and state the grain of the joined table.

# TODO 14c: perform the join. Consider outer vs inner carefully -- and check for
#           row multiplication if your keys are not unique on both sides
#           (you already know from Q4 whether they are).

# TODO 14d: actual vs predicted. Explain the gap. A large gap usually means either
#           duplicate keys or a misunderstanding of the grain -- diagnose which.

# TODO 14e: how many locations enrol but never update? Never enrol but update?
#           These are the rows an inner join would have thrown away.

### ✍️ Your answer — Q14

_Replace this text with your written justification. Every judgement call (a threshold, a merge rule, a dropped row, a bin width) needs a sentence explaining why. Unjustified choices are marked as if the choice were wrong._

## Q15. Convert counts into rates

Bring in state population from an **external** source (Census 2011, or a projection — cite
whichever you use and note its age). Compute per-capita enrolment and update rates by state.

Present the **top ten states by raw count** beside the **top ten by rate**. Explain why the two
lists differ, and state which one answers the question *"where is Aadhaar coverage weakest."*

**Deliverable:** A cited population table, both top-ten lists side by side, and a direct answer to the coverage question.

In [ ]:
# TODO 15a: source state populations. Census 2011 is the last full census; note that
#           it is well over a decade old and say what that does to your rates.
#           Enter it as a small DataFrame with a citation in a comment.
population = pd.DataFrame(columns=["state", "population", "source", "year"])

# TODO 15b: join to your canonicalised state names from Q5. Report any state that
#           failed to match -- a silent failure here quietly drops a whole state.

# TODO 15c: compute per-capita rates.

# TODO 15d: top 10 by raw count vs top 10 by rate, side by side.

# TODO 15e: answer directly -- where is coverage weakest, and how confident are you
#           given the age of the denominator?

### ✍️ Your answer — Q15

_Replace this text with your written justification. Every judgement call (a threshold, a merge rule, a dropped row, a bin width) needs a sentence explaining why. Unjustified choices are marked as if the choice were wrong._

## Section E — Anomaly Detection

## Q16. Separate system glitches from real events

Build a daily national time series per table and detect anomalies using **two methods** — one
assuming normality (z-score), one robust (median absolute deviation or an IQR rule). Compare
what each flags.

Then classify the anomalies: **reporting outages** (unexpected zeros), **catch-up spikes** (a
zero day followed by a doubled day), and **genuine surges**.

Give the diagnostic test that separates them, and say which category dominates.

**Deliverable:** A comparison of what each method flags, a classified anomaly table, and a statement of which category dominates.

In [ ]:
# TODO 16a: daily national totals per table.

# TODO 16b: flag anomalies two ways. Note that z-score assumes normality -- check
#           whether that assumption holds here before relying on it.

# TODO 16c: compare the two flag sets. Where do they disagree, and why?

# TODO 16d: classify. The catch-up signature is specific: a trough immediately
#           followed by a compensating peak, where the PAIR sums to roughly the
#           local baseline x 2. Write a test for that signature.

# TODO 16e: which category dominates? What does that say about data quality?

### ✍️ Your answer — Q16

_Replace this text with your written justification. Every judgement call (a threshold, a merge rule, a dropped row, a bin width) needs a sentence explaining why. Unjustified choices are marked as if the choice were wrong._

## Section F — Visualization

Four questions. Chart quality is graded: axis labels, units, readable scales, and a title that
states the finding rather than naming the variables.

## Q17. Map it two ways

Produce a choropleth of India by state for **total enrolments**, and beside it the same map for
**per-capita enrolment rate** from Q15.

Use a sequential colour scale appropriate to each and state your binning choice. Write two
sentences on what the raw-count map implies that is misleading.

Confirm every state in your shapefile matched a state in your canonicalised data — and report
any that did not.

**Deliverable:** Two choropleths side by side, a stated binning rule, a shapefile match report, and two sentences on the misleading reading.

In [ ]:
# Shapefile / GeoJSON options (any is acceptable, cite what you use):
#   - geopandas + a public India state GeoJSON
#   - plotly.express.choropleth with a GeoJSON featureidkey
# Colab: !pip install -q geopandas   (may take a minute)

# TODO 17a: load boundaries. Reconcile shapefile state names against your Q5 canonical
#           names -- this is a THIRD naming convention and it will not match cleanly.
#           Report unmatched states in BOTH directions.

# TODO 17b: choropleth of raw totals.

# TODO 17c: choropleth of per-capita rate, same projection, same figure size.

# TODO 17d: state your binning rule (quantile / equal-interval / natural breaks) and
#           why. The choice changes which states look alarming.

### ✍️ Your answer — Q17

_Replace this text with your written justification. Every judgement call (a threshold, a merge rule, a dropped row, a bin width) needs a sentence explaining why. Unjustified choices are marked as if the choice were wrong._

## Q18. Show the reporting artifact

Build a **small-multiples** figure of monthly enrolment for the twelve largest states.

Choose linear or log scale and defend the choice — state sizes span orders of magnitude.

The figure must make the **data-coverage effect from Q11 visible** rather than hiding it behind
an apparently smooth growth curve. Annotate at least one specific feature directly on the chart.

**Deliverable:** A 12-panel small-multiples figure with a defended scale choice and at least one direct annotation.

In [ ]:
# TODO 18a: monthly enrolment per state; select the 12 largest by total.

# TODO 18b: small multiples -- shared axes or free axes? The choice determines whether
#           readers compare LEVELS or SHAPES. State which comparison you are enabling.

# TODO 18c: make the coverage effect visible. Options: a second line or shaded band
#           showing reporting-pincode count per month, or panels shaded where coverage
#           was incomplete.

# TODO 18d: annotate at least one specific feature directly on the chart (ax.annotate),
#           not in the caption.

### ✍️ Your answer — Q18

_Replace this text with your written justification. Every judgement call (a threshold, a merge rule, a dropped row, a bin width) needs a sentence explaining why. Unjustified choices are marked as if the choice were wrong._

## Q19. Show the long tail

Enrolment counts across ~19,000 pincodes will be severely right-skewed, with a small number of
locations dominating.

Produce **one** figure that communicates that concentration honestly — a log-scale
distribution, a ranked curve, or a Lorenz curve with a Gini coefficient.

Report what share of total enrolments comes from the **top 1% of pincodes**, and state whether
that concentration is surprising given where people live.

**Deliverable:** One figure, the top-1% share as a number, and a judgement on whether the concentration is surprising.

In [ ]:
# TODO 19a: total enrolments per pincode. Describe the skew numerically before plotting
#           (mean vs median, skewness, the 99th percentile vs the max).

# TODO 19b: build ONE figure. If you choose a Lorenz curve, compute and report Gini.
#           If you choose a log-scale histogram, justify the bin count explicitly.

# TODO 19c: top 1% share, top 10% share, and the share of pincodes needed to reach 50%
#           of all enrolments.

# TODO 19d: interpret. India's population is itself highly concentrated in cities.
#           Is this distribution more concentrated than population, or just as much?

### ✍️ Your answer — Q19

_Replace this text with your written justification. Every judgement call (a threshold, a merge rule, a dropped row, a bin width) needs a sentence explaining why. Unjustified choices are marked as if the choice were wrong._

## Q20. Critique and revise

Take your strongest figure from Q17–Q19 and write a critique:

1. One conclusion a policymaker could **wrongly** draw from it.
2. One design decision of yours that shaped that conclusion.
3. One thing this dataset **cannot** tell you no matter how it is plotted — think about what
   aggregation to pincode-day counts permanently discards.

Then produce a **revised figure** addressing point 1, and explain what you traded away to fix it.

**Deliverable:** A three-part written critique, a revised figure, and a statement of the trade-off.

In [ ]:
# TODO 20a: reproduce your chosen figure here so the critique is self-contained.

# TODO 20b: write the three-part critique in the answer cell below.
#           For point 3, consider: this data is aggregated counts per pincode-day-ageband.
#           Individual-level questions -- repeat updates by the same person, time between
#           a person's enrolment and their first update, household patterns -- are
#           permanently unanswerable. What else is gone?

# TODO 20c: produce the revised figure.

# TODO 20d: name the trade-off. Every fix costs something -- usually simplicity.

### ✍️ Your answer — Q20

_Replace this text with your written justification. Every judgement call (a threshold, a merge rule, a dropped row, a bin width) needs a sentence explaining why. Unjustified choices are marked as if the choice were wrong._